# Pipeline de Extracción de Datos de *Open-Meteo*

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener datos históricos climáticos de *Open-Meteo*. El objetivo es consolidar una base de datos sobre diversas mediciones climaticas en las 12 ciudades españolas.

Obtenemos los datos de la API de *Open-Meteo*. Para ello, debemos definir el rango de tiempo sobre el que queremos los datos (en horas), la longitud y latitud de la ciudad deseada y, finalmente, seleccionar las variables de las que pretendemos tener los datos. 

Requerimos de estos datos para entender el comportamiento de los contaminantes, pues es imprescindible considerar las condiciones meteorológicas. Por consiguiente, en esta sección seguimos las siguientes consideraciones:

* **Geolocalización precisa**: empleamos coordenadas exactas (latitud/longitud) de las ciudades para asegurar la coincidencia geográfica con las estaciones de la EEA.

* **Sincronización temporal**: configuramos la zona horaria en UTC para garantizar un alineamiento perfecto al fusionar estos datos con las mediciones de calidad del aire.

* **Estructura espejo**: guardamos los datos climáticos siguiendo la misma jerarquía de carpetas por país que los datos de contaminantes, facilitando la integración final del dataset.

Además, las variables que contendrá cada *dataset* por país son:

1. *city*: ciudad a la que pertenecen los datos.
2. *time*: momento (día-mes-año-hora) en que se toman los datos.
3. *temperature_2m*: temperatura a 2 metros en grados (°C).
4. *relative_humidity_2m*: humedad relativa a 2 metros, en porcentaje.
5. *precipitation*: precipitación total (lluvia + nieve), en mm (milímetros).
6. *rain*: lluvia en mm.
7. *snowfall*: lluvia de la nieve en cm.
8. *surface_pressure*: presión atmosférica en superficie, en hPa (hectopascal).
9. *cloudcover*: cobertura nubosa total, en porcentaje.
9. *windspeed_10m*: velocidad del viento a 10 metros, en km/h.
11. *shortwave_radiation*: radiación solar de onda corta, en $W/m^2$.
12. *boundary_layer_height*: capa PBLH (*Planetary Boundary Layer Height*) o **Altura de la Capa Límite Planetaria** en metros. Se podría comparar con un techo que tenemos todos los seres humanos y que permite que los gases producidos por vehículos o fábricas se vaya, por lo que se concentran.

In [ ]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
import requests           # Para realizar peticiones a la API climática.
import os    
import pandas as pd        # Procesamiento de las series temporales meteorológicas.
from pathlib import Path   # Gestión de rutas y directorios de forma robusta.
from tqdm import tqdm     # Visualización del progreso durante la descarga masiva.
import time

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# Endpoint de datos históricos (Archive API) de Open-Meteo.
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"


# Definimos la ruta base de datasets.
# La carpeta "datasets" está un nivel por encima de la carpeta actual del proyecto.
BASE_PATH = Path("..", "..") / "datasets"

# Definición del directorio de salida específico para datos climáticos.
BASE_DIR = BASE_PATH / "archivos_clima"

# Creamos la carpeta si todavía no existe.
BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Carpeta de datos climáticos preparada en: {BASE_DIR}")

📂 Carpeta de datos climáticos preparada en: ..\..\datasets\archivos_clima


In [3]:
# Rango temporal: debe coincidir con el periodo de los datos de la EEA (2013-2024).
START_DATE = "2013-01-01"
END_DATE   = "2024-12-31"

In [4]:
# ==============================================================================
# CATÁLOGO DE COORDENADAS
# ==============================================================================
# Diccionario que mapea los nombres con las coordenadas (Latitud, Longitud) exactas obtenidas del notebook "Ciudades" de las ciudades españolas analizadas en el bloque de la EEA.

locations = {
    "Madrid": (40.41694, -3.70333),
    "Barcelona": (41.3825, 2.17694),
    "Bilbao": (43.26306, -2.935),
    "Sevilla": (37.38861, -5.99556),
    "Valencia": (39.47, -0.37639),
    "Valladolid": (41.65198, -4.72856),
    "Murcia": (37.98333, -1.13028),
    "Santa Cruz de Tenerife": (28.46667, -16.25),
    "A Coruña": (43.337386, -8.40003),
    "Zaragoza": (41.65, -0.88333),
    "Alicante/Alacant": (38.34528, -0.48306),
    "Albacete": (38.99556, -1.85583)
}

In [ ]:
# ==============================================================================
# EJECUCIÓN DEL PIPELINE DE DESCARGA POR CIUDAD
# ==============================================================================

# Se itera sobre el diccionario `locations`, que contiene pares:
# {nombre_ciudad: (latitud, longitud)}.
# tqdm se utiliza para visualizar una barra de progreso durante el proceso
# de descarga de datos climáticos para cada ciudad.
for city, (lat, lon) in tqdm(locations.items(), desc="Downloading climate data"):
    
    # --------------------------------------------------------------------------
    # CREACIÓN DE ESTRUCTURA DE CARPETAS
    # --------------------------------------------------------------------------
    
    # Limpieza del nombre de la ciudad para evitar caracteres problemáticos
    # en sistemas de archivos (espacios o "/").
    # Ejemplo: "Santa Cruz" -> "Santa_Cruz"
    city_clean = city.replace(" ", "_").replace("/", "_")
    
    # Se define la carpeta específica para almacenar los datos de esa ciudad.
    # BASE_DIR es el directorio raíz donde se guardarán todos los datos.
    city_dir = BASE_DIR / city_clean
    
    # Se crea la carpeta si no existe.
    # parents=True permite crear carpetas intermedias automáticamente.
    # exist_ok=True evita errores si la carpeta ya existe.
    city_dir.mkdir(parents=True, exist_ok=True)
    
    # Se define el archivo de salida donde se almacenarán los datos descargados.
    # Cada ciudad tendrá un único archivo consolidado.
    output_file = city_dir / f"open-meteo-{city_clean}.csv"

    # --------------------------------------------------------------------------
    # CONTROL DE DESCARGAS DUPLICADAS
    # --------------------------------------------------------------------------

    # Si el archivo ya existe, se omite la descarga.
    # Esto evita:
    # - re-descargar datos innecesariamente
    # - generar tráfico adicional a la API
    # - provocar errores de rate-limit (HTTP 429)
    if output_file.exists():
        continue
 
    # --------------------------------------------------------------------------
    # DEFINICIÓN DE VARIABLES CLIMÁTICAS A DESCARGAR
    # --------------------------------------------------------------------------

    # Diccionario de parámetros que se enviarán a la API de Open-Meteo.
    # Incluye coordenadas geográficas, rango temporal y variables meteorológicas.
    params = {
        "latitude": lat,     # Latitud de la ciudad
        "longitude": lon,    # Longitud de la ciudad
        "start_date": START_DATE,  # Fecha inicial del dataset
        "end_date": END_DATE,      # Fecha final del dataset
        
        # Variables meteorológicas solicitadas con resolución horaria
        # (se envían como string separado por comas)
        "hourly": (
            "temperature_2m,"         # Temperatura a 2 metros
            "relative_humidity_2m,"   # Humedad relativa
            "precipitation,"          # Precipitación total
            "rain,"                   # Lluvia
            "snowfall,"               # Nieve
            "surface_pressure,"       # Presión atmosférica
            "cloudcover,"             # Cobertura nubosa
            "windspeed_10m,"          # Velocidad del viento a 10 m
            "shortwave_radiation,"    # Radiación solar
            "boundary_layer_height"   # Altura de la capa límite atmosférica
        ),
        
        # Se fija la zona horaria en UTC para mantener consistencia
        # con otros datasets ambientales o de contaminación.
        "timezone": "UTC"
    }

    # --------------------------------------------------------------------------
    # CONTROL DE REINTENTOS (RETRY LOGIC)
    # --------------------------------------------------------------------------

    success = False      # Indicador de descarga completada
    retries = 0          # Contador de reintentos
    max_retries = 3      # Máximo número de intentos por ciudad

    # Mientras no se haya completado la descarga y no se exceda el número
    # máximo de reintentos, se seguirá intentando realizar la solicitud.
    while not success and retries < max_retries:
        try:
            # ------------------------------------------------------------------
            # SOLICITUD HTTP A LA API
            # ------------------------------------------------------------------

            # Se realiza una petición GET a la API con los parámetros definidos.
            # requests se encarga de convertir automáticamente el diccionario
            # `params` en parámetros de URL.
            response = requests.get(BASE_URL, params=params)
            
            # Si la API devuelve código 429 significa que se ha alcanzado
            # el límite de peticiones permitido (rate limit).
            if response.status_code == 429:
                print(f"\n⚠️ Límite alcanzado. Esperando 60 segundos para {city}...")
                
                # Se espera 60 segundos antes de intentar nuevamente.
                time.sleep(60)
                retries += 1
                continue

            # Si la respuesta HTTP contiene error (4xx o 5xx),
            # se lanza automáticamente una excepción.
            response.raise_for_status()

            # Se convierte la respuesta JSON en un objeto Python (diccionario).
            data = response.json()

            # ------------------------------------------------------------------
            # TRANSFORMACIÓN DEL JSON A DATAFRAME
            # ------------------------------------------------------------------

            # La API devuelve los datos horarios dentro de la clave "hourly".
            if "hourly" in data:

                # Se transforma el bloque de datos horarios en un DataFrame.
                df = pd.DataFrame(data["hourly"])

                # Se añaden columnas identificadoras para facilitar
                # posteriores integraciones con otros datasets.
                # Estas columnas permiten hacer joins con datasets
                # de contaminación, población, etc.
                df.insert(0, "city", city)
                df.insert(1, "latitude", lat)
                df.insert(2, "longitude", lon)

                # ------------------------------------------------------------------
                # EXPORTACIÓN DE DATOS
                # ------------------------------------------------------------------

                # Se guarda el DataFrame final en formato CSV.
                # index=False evita escribir el índice de pandas como columna.
                df.to_csv(output_file, index=False)

                success = True

                # Pausa breve para evitar saturar la API con demasiadas solicitudes.
                time.sleep(2) 
            
        # ----------------------------------------------------------------------
        # MANEJO DE ERRORES
        # ----------------------------------------------------------------------

        except Exception as e:
            print(f"\n❌ Error en {city}: {e}")
            retries += 1

            # Espera breve antes de volver a intentar.
            time.sleep(5)

# --------------------------------------------------------------------------
# FINALIZACIÓN DEL PROCESO
# --------------------------------------------------------------------------

# Mensaje final indicando que el pipeline ha terminado
# y dónde se han guardado los archivos descargados.
print(f"\n✅ Proceso completado. Revisa tu carpeta: {BASE_DIR}")


✅ Proceso completado. Revisa tu carpeta: ..\..\datasets\archivos_clima


Los archivos descargados directamente de la API meteorológica contienen las columnas de latitud y longitud que no son de interés, por lo que se eliminan.

In [ ]:
# ==============================================================================
# LIMPIEZA DE COLUMNAS (LAT/LON)
# ==============================================================================

# Recorremos todas las carpetas de ciudades
for city_folder in BASE_DIR.iterdir():
    
    if city_folder.is_dir(): 
        # Buscamos archivos CSV dentro de cada carpeta
        for csv_file in city_folder.glob("*.csv"):
            try:
                # Cargamos el CSV (usamos sep=None para que detecte si es coma o punto y coma automáticamente)
                df = pd.read_csv(csv_file, sep=None, engine='python')
                
                # Eliminamos las columnas de latitud y longitud si existen
                # errors='ignore' evita que el código falle si alguna ya fue borrada antes
                cols_to_drop = ['latitude', 'longitude']
                df = df.drop(columns=cols_to_drop, errors='ignore')
                
                # Guardamos
                df.to_csv(csv_file, index=False, encoding='utf-8-sig')
                
                
            except Exception as e:
                print(f"❌ Error procesando {csv_file.name}: {e}")

print("\n🚀 ¡Listo! Todos los CSV han sido actualizados y limpiados.")



🚀 ¡Listo! Todos los CSV han sido actualizados y limpiados.
